# Session 1: a rain cell drifts with the wind
**A. Real-world problem.** A weather radar sends out a picture of rain every 5 minutes.
A rain cell (a blob of heavy rain) in one picture is in a new place in the next picture.
If we measure how far it moved, we know the wind that is carrying it. Then we can say when
it will reach a town in the next hour. This is called a *nowcast* (a forecast for the next
hour or two). Safety note: this is a teaching model, not a warning system.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
PX_KM = 1.0          # one radar pixel is 1 km wide
DT_MIN = 5.0         # a new radar picture every 5 minutes
N = 100              # the picture is 100 x 100 pixels (100 km x 100 km)

## Step 1: a radar picture is a grid of numbers
Each pixel holds a rain rate in mm/h (millimetres of rain per hour). One rain cell is
a smooth hill of rain: strongest at the centre, weaker outside. We model it as a Gaussian
bell with width `sigma` (km) and centre rain rate `peak` (mm/h).

In [ ]:
def gauss_cell(cx, cy, sigma, peak, n=N):
    """A rain cell centred at pixel (cx, cy). Returns an n x n array (rows = y, columns = x) in mm/h."""
    y, x = np.mgrid[0:n, 0:n]
    return peak * np.exp(-((x - cx) ** 2 + (y - cy) ** 2) / (2 * sigma ** 2))


def centroid(field):
    """Rain-weighted centre (cx, cy) of a picture, in pixels."""
    y, x = np.mgrid[0:field.shape[0], 0:field.shape[1]]
    total = field.sum()
    return float((field * x).sum() / total), float((field * y).sum() / total)

## Step 2: how fast is the cell moving?
Compare two pictures taken 5 minutes apart. If the centre moved `dx` pixels east and `dy`
pixels north, then 1 pixel = 1 km and 5 minutes = 1/12 hour give
    wind = pixels moved * 1 km / (1/12 h) = 12 * pixels moved   (km/h).

In [ ]:
def wind_from_shift(dx_px, dy_px, px_km=PX_KM, dt_min=DT_MIN):
    """Wind components (u east, v north) in km/h, speed in km/h and direction in degrees (0 = east, 90 = north)."""
    hours = dt_min / 60.0
    u, v = dx_px * px_km / hours, dy_px * px_km / hours
    return u, v, float(np.hypot(u, v)), float(np.degrees(np.arctan2(v, u)))

## Step 3: when will it reach the town?
Assume the wind stays the same (the key assumption of this whole repo). The cell centre then
moves in a straight line at constant speed. The closest the centre comes to the town, and
when, follow from simple geometry. Rain at the town is the bell evaluated at the distance
from the centre.

In [ ]:
def cell_position(c0, wind, t_min):
    """Centre (x, y) in km after t_min minutes. c0 = (x, y) now, wind = (u, v) in km/h."""
    return c0[0] + wind[0] * t_min / 60.0, c0[1] + wind[1] * t_min / 60.0


def closest_approach(c0, wind, town):
    """(time in minutes, distance in km) of the closest approach of the cell centre to the town."""
    rx, ry = town[0] - c0[0], town[1] - c0[1]
    sp2 = wind[0] ** 2 + wind[1] ** 2
    if sp2 == 0:
        return float("inf"), float(np.hypot(rx, ry))
    t_h = max(0.0, (rx * wind[0] + ry * wind[1]) / sp2)      # no negative times: the cell moves forward only
    cx, cy = cell_position(c0, wind, t_h * 60.0)
    return t_h * 60.0, float(np.hypot(town[0] - cx, town[1] - cy))


def rain_at(point, centre, sigma, peak):
    """Rain rate (mm/h) at a point, from a Gaussian cell."""
    return peak * np.exp(-((point[0] - centre[0]) ** 2 + (point[1] - centre[1]) ** 2) / (2 * sigma ** 2))


def rain_edge_radius(sigma, peak, threshold):
    """Distance (km) from the centre at which the rain falls to `threshold` mm/h; 0 if the peak is below it."""
    return 0.0 if peak <= threshold else float(sigma * np.sqrt(2 * np.log(peak / threshold)))


def first_rain_time(c0, wind, town, sigma, peak, threshold):
    """First time (minutes) at which the town gets at least `threshold` mm/h, or None if it never does."""
    r = rain_edge_radius(sigma, peak, threshold)
    if r == 0.0:
        return None
    ax, ay = c0[0] - town[0], c0[1] - town[1]
    a = (wind[0] ** 2 + wind[1] ** 2) / 3600.0                 # per minute squared
    b = 2 * (ax * wind[0] + ay * wind[1]) / 60.0
    c = ax ** 2 + ay ** 2 - r ** 2
    if c <= 0:
        return 0.0                                             # already raining at the town
    if a == 0:
        return None
    disc = b * b - 4 * a * c
    if disc < 0:
        return None                                            # the edge of the cell never reaches the town
    t = (-b - np.sqrt(disc)) / (2 * a)
    return float(t) if t >= 0 else None

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    c0, sigma, peak, thr = (20.0, 50.0), 6.0, 40.0, 5.0
    town = (56.0, 50.0)
    u, v, sp, ang = wind_from_shift(3, 0)
    print(f"cell moved 3 px in 5 min -> wind {u:.0f} km/h east (speed {sp:.0f} km/h, direction {ang:.0f} deg)")
    t_c, miss = closest_approach(c0, (u, v), town)
    print(f"centre is closest at {t_c:.0f} min, miss distance {miss:.1f} km")
    print(f"first rain >= {thr:.0f} mm/h at the town after {first_rain_time(c0, (u, v), town, sigma, peak, thr):.1f} min")
    tilt = np.radians(20)
    w2 = (sp * np.cos(tilt), sp * np.sin(tilt))
    t2, miss2 = closest_approach(c0, w2, town)
    print(f"wind turned by 20 deg: closest at {t2:.1f} min, miss {miss2:.1f} km, "
          f"peak rain at town {rain_at(town, cell_position(c0, w2, t2), sigma, peak):.2f} mm/h")

    fig, axs = plt.subplots(1, 4, figsize=(11, 3.2))
    for ax, t in zip(axs, (0, 20, 40, 60)):
        cx, cy = cell_position(c0, (u, v), t)
        ax.imshow(gauss_cell(cx, cy, sigma, peak), origin="lower", cmap="Blues", vmin=0, vmax=40)
        ax.plot(*town, "r*", ms=12)
        ax.set_title(f"t = +{t} min")
        ax.set_xlabel("x (km)")
    axs[0].set_ylabel("y (km)")
    fig.suptitle("A rain cell drifts east at 36 km/h; red star = town")
    fig.tight_layout()
    fig.savefig(OUT / "drifting_cell.png", dpi=110)

    ts = np.linspace(0, 120, 241)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    for wind, lab in (((u, v), "wind straight at the town"), (w2, "wind turned by 20 degrees")):
        ax2.plot(ts, [rain_at(town, cell_position(c0, wind, t), sigma, peak) for t in ts], label=lab)
    ax2.axhline(thr, color="gray", ls=":")
    ax2.set_xlabel("time from now (min)")
    ax2.set_ylabel("rain rate at the town (mm/h)")
    ax2.set_title("Forecast rain at the town")
    ax2.legend()
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "rain_at_town.png", dpi=120)

## Try it
Do the coding tasks, the README questions (F), then the checkpoint.

## Coding tasks
1. **Easy.** Change the cell speed in `lesson.py` to 5 pixels per frame and the town to 45 km away. Predict the arrival time before you run it.
2. **Stretch.** Write `eta_for_wind(u, v)` that uses `closest_approach` to print the arrival time and miss distance for the winds (36, 0) and (30, 12) km/h and a town 36 km east of the cell.
3. **Explain.** Explain in your own words why turning the wind by 20 degrees made such a big difference to the town even though the path only moved 12.3 km sideways.

## Assessment
- **A. Conceptual.** Why do we need two radar pictures to get the wind, and why do we assume the wind stays the same during the forecast?
- **B. Numerical.** A cell moves 5 pixels east in 5 minutes. A town is 45 km east of its centre. What is the wind and when does the centre arrive?
- **C. Graph and simulation interpretation.** In the graph of rain at the town, the wind straight at the town and the wind turned by 20 degrees give very different peaks. Read both peaks and explain the difference.
- **D. Design and optimisation challenge.** Design: a town wants at least 20 minutes of warning before rain of 5 mm/h or more, for a cell moving straight at it at 36 km/h. How far away must the cell centre be at the moment the forecast is issued?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("the picture is 100 x 100 pixels of 1 km, one picture every 5 minutes",
                   L.N == 100 and L.PX_KM == 1.0 and L.DT_MIN == 5.0))
    f = L.gauss_cell(50, 50, 6.0, 40.0)
    checks.append(("a cell has its peak rain rate at its centre (40 mm/h)", f.shape == (100, 100) and abs(f[50, 50] - 40.0) < 1e-9 and f.max() == f[50, 50]))
    cx, cy = L.centroid(L.gauss_cell(45.0, 38.0, 6.0, 40.0))
    checks.append(("the rain-weighted centre of a cell is where we put it", abs(cx - 45.0) < 1e-6 and abs(cy - 38.0) < 1e-6))
    u, v, sp, ang = L.wind_from_shift(3, 0)
    checks.append(("worked example: 3 pixels east in 5 minutes is 36 km/h towards the east", abs(u - 36.0) < 1e-9 and abs(v) < 1e-9 and abs(sp - 36.0) < 1e-9 and abs(ang) < 1e-9))
    u1, _, _, _ = L.wind_from_shift(1, 0)
    checks.append(("hand calculation: 1 pixel in 5 minutes is 12 km/h", abs(u1 - 12.0) < 1e-9))
    t, miss = L.closest_approach((20.0, 50.0), (36.0, 0.0), (56.0, 50.0))
    checks.append(("worked example: the cell centre reaches a town 36 km away after 60 minutes", abs(t - 60.0) < 1e-9 and abs(miss) < 1e-9))
    r = L.rain_edge_radius(6.0, 40.0, 5.0)
    checks.append(("worked example: the 5 mm/h edge of the cell is 12.24 km from its centre", abs(r - 12.2361) < 1e-3))
    ft = L.first_rain_time((20.0, 50.0), (36.0, 0.0), (56.0, 50.0), 6.0, 40.0, 5.0)
    checks.append(("worked example: rain of 5 mm/h reaches the town after 39.6 minutes", ft is not None and abs(ft - 39.6) < 0.05))
    p = L.cell_position((20.0, 50.0), (36.0, 0.0), ft)
    checks.append(("at that time the town really is 5 mm/h", abs(L.rain_at((56.0, 50.0), p, 6.0, 40.0) - 5.0) < 1e-6))
    w2 = (36.0 * np.cos(np.radians(20)), 36.0 * np.sin(np.radians(20)))
    t2, miss2 = L.closest_approach((20.0, 50.0), w2, (56.0, 50.0))
    checks.append(("a wind turned by 20 degrees misses the town by 12.3 km (36 sin 20)", abs(miss2 - 36.0 * np.sin(np.radians(20))) < 1e-9 and abs(miss2 - 12.3) < 0.05))
    peak2 = L.rain_at((56.0, 50.0), L.cell_position((20.0, 50.0), w2, t2), 6.0, 40.0)
    checks.append(("with that miss the town only sees 4.87 mm/h at best, just under 5", abs(peak2 - 4.87) < 0.01 and peak2 < 5.0))
    checks.append(("so the 5 mm/h rain never arrives", L.first_rain_time((20.0, 50.0), w2, (56.0, 50.0), 6.0, 40.0, 5.0) is None))
    checks.append(("zero wind: the cell never arrives", L.closest_approach((0.0, 0.0), (0.0, 0.0), (10.0, 0.0))[0] == float("inf")
                   and L.first_rain_time((0.0, 0.0), (0.0, 0.0), (50.0, 0.0), 6.0, 40.0, 5.0) is None))
    checks.append(("a cell moving away from the town is closest right now (t = 0)", L.closest_approach((20.0, 50.0), (-36.0, 0.0), (56.0, 50.0))[0] == 0.0))
    checks.append(("a town that is already raining on gets time 0", L.first_rain_time((50.0, 50.0), (10.0, 0.0), (52.0, 50.0), 6.0, 40.0, 5.0) == 0.0))
    checks.append(("a cell weaker than the threshold never gives it", L.first_rain_time((20.0, 50.0), (36.0, 0.0), (56.0, 50.0), 6.0, 4.0, 5.0) is None))
    u5 = L.wind_from_shift(5, 0)[0]
    t5, _ = L.closest_approach((0.0, 0.0), (u5, 0.0), (45.0, 0.0))
    checks.append(("exercise: 5 pixels in 5 minutes is 60 km/h and a town 45 km away is reached in 45 minutes", abs(u5 - 60.0) < 1e-9 and abs(t5 - 45.0) < 1e-9))
    r5 = L.rain_edge_radius(6.0, 40.0, 5.0)
    ft20 = L.first_rain_time((56.0 - 36.0 * 20 / 60 - r5, 50.0), (36.0, 0.0), (56.0, 50.0), 6.0, 40.0, 5.0)
    checks.append(("design: for 20 minutes of warning the cell centre must be 12 + 12.24 = 24.24 km away at 36 km/h",
                   abs(36.0 * 20 / 60 + r5 - 24.24) < 0.01 and abs(ft20 - 20.0) < 1e-6))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
